# Connect to Drive

In [ ]:
import os

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.exists("src"):
        !git clone https://github.com/HenriqueSchmitz/world-models-implementation temp_repo
        !mv temp_repo/* .
        !rm -rf temp_repo
        !sudo apt-get install -y swig
        %pip install -q -r requirements.txt

In [ ]:
import os
from src.utils.colab import is_environment_colab_notebook
from src.utils.secrets import get_secret

project_folder = get_secret("worldModelsFolderPath") if is_environment_colab_notebook() else "./"
settings_path = os.path.join(project_folder, "settings.json")
settings_path = settings_path if os.path.exists(settings_path) else  "./settings.json"
world_model_folder = os.path.join(project_folder, "weights/worldmodel")
world_model_folder = world_model_folder if os.path.exists(world_model_folder) else  "./weights/worldmodel"
output_folder = os.path.join(project_folder, "weights/controller")
os.makedirs(output_folder, exist_ok=True)

# Training Settings

In [ ]:
from src.utils.logging import get_logger
LOG_LEVEL = "INFO"
logger = get_logger(LOG_LEVEL)

In [ ]:
import json

with open(settings_path, "r") as settings_file:
    settings = json.load(settings_file)
    REPRESENTATION_DIM = settings["vae"]["model"]["representation_dim"]
    RNN_HIDDEN_DIM = settings["world_model"]["model"]["hidden_dim"]
    ACTION_DIM = settings["controller"]["model"]["action_dim"]

    NUM_GENERATIONS = settings["controller"]["train"]["num_generations"]
    POPULATION_SIZE = settings["controller"]["train"]["population_size"]
    STEPS_PER_ROLLOUT = settings["controller"]["train"]["steps_per_rollout"]
    ROLLOUTS_PER_SOLUTION = settings["controller"]["train"]["rollouts_per_solution"]
    SIGMA_INIT = settings["controller"]["train"]["sigma_init"]

WANDB_PROJECT = "world-models-paper"
WANDB_RUN_NAME = "controller"

def log_settings():
    logger.info(f"REPRESENTATION_DIM: {REPRESENTATION_DIM}")
    logger.info(f"RNN_HIDDEN_DIM: {RNN_HIDDEN_DIM}")
    logger.info(f"ACTION_DIM: {ACTION_DIM}")
    logger.info("========================================")
    logger.info(f"NUM_GENERATIONS: {NUM_GENERATIONS}")
    logger.info(f"POPULATION_SIZE: {POPULATION_SIZE}")
    logger.info(f"STEPS_PER_ROLLOUT: {STEPS_PER_ROLLOUT}")
    logger.info(f"ROLLOUTS_PER_SOLUTION: {ROLLOUTS_PER_SOLUTION}")
    logger.info(f"SIGMA_INIT: {SIGMA_INIT}")
    logger.info("========================================")
    logger.info(f"WANDB_PROJECT: {WANDB_PROJECT}")
    logger.info(f"WANDB_RUN_NAME: {WANDB_RUN_NAME}")

log_settings()

# Setup

In [ ]:
from src.models.controller import Controller
from src.models.simulation_worldmodel import SimulationWorldModel
from src.training.controller_trainer import ControllerEvolutionaryTrainer
from src.utils.torch import get_device
from src.utils.secrets import get_secret

In [ ]:
DEVICE = get_device()

# Train

In [ ]:
world_model_path = os.path.join(world_model_folder, f"model.pth")
simulation_worldmodel = SimulationWorldModel(worldmodel_path=world_model_path,
                                             settings_path=settings_path,
                                             device=DEVICE,
                                             batch_size=ROLLOUTS_PER_SOLUTION,
                                             logger=logger)

In [ ]:
model = Controller(observation_dim=REPRESENTATION_DIM,
                   hidden_dim=RNN_HIDDEN_DIM,
                   action_dim=ACTION_DIM,
                   device=DEVICE)

In [ ]:
try:
    wandb_setup = {
        "api_key": get_secret('wandbApiKey'),
        "project": WANDB_PROJECT,
        "run_name": WANDB_RUN_NAME,
        "config": {
            "generations": NUM_GENERATIONS,
            "population_size": POPULATION_SIZE,
            "steps_per_rollout": STEPS_PER_ROLLOUT,
            "rollouts_per_solution": ROLLOUTS_PER_SOLUTION,
            "sigma_init": SIGMA_INIT,
        }
    }
except Exception as e:
    logger.error(f"Ignoring wandb logging, could not retrieve wandbApiKey secret. Error: {str(e)}")
    wandb_setup = None

In [ ]:
trainer = ControllerEvolutionaryTrainer(model=model,
                                        weights_folder=output_folder,
                                        num_generations=NUM_GENERATIONS,
                                        population_size=POPULATION_SIZE,
                                        simulation_world_model=simulation_worldmodel,
                                        steps_per_rollout=STEPS_PER_ROLLOUT,
                                        rollouts_per_solution=ROLLOUTS_PER_SOLUTION,
                                        sigma_init=SIGMA_INIT,
                                        load_checkpoint=True,
                                        device=DEVICE,
                                        wandb_setup=wandb_setup,
                                        logger=logger)
                                        

In [ ]:
# Repeat logging of important information so it is available on wandb run
log_settings()
get_device(logger)

In [ ]:
trainer.train()